# Assignment 1: Opinion Network Formation

In this notebook, we are constructing a respondent-based network from the survey dataset, `Survey_Results_UC.csv`[cite: 1]. Conceptually, this process shares the same underlying logic used when mapping structural connectomes or calculating distance metrics for a k-nearest neighbors classifier: we are transforming behavioral data into a multidimensional feature space to identify structural clusters. 

Here, each respondent serves as a distinct node, and the edges will be determined by the mathematical similarity of their survey answers across the four domains (Technology, Education, Ethics, Environment)[cite: 1].

### Step 1: Vectorization and Data Preprocessing
The raw data consists of text-based Likert-scale responses. Before we can compute a similarity matrix or isolate sub-graphs, we must encode these string values into a continuous numerical format. 

In the execution block below, we will:
1. Load `Survey_Results_UC.csv` into a pandas DataFrame.
2. Construct a mapping dictionary to convert ordinal text (e.g., "Strongly Disagree" to "Strongly Agree") into integer values (1 through 5). We handle edge cases like "No Comments" by imputing a neutral weight (3) to prevent matrix sparsity.
3. Separate the numerical feature matrix from the respondent IDs to prepare for the cosine similarity operations in the next step.

In [ ]:
import pandas as pd
import numpy as np

# 1. Load the dataset
df = pd.read_csv('Survey_Results_UC.csv')

# 2. Define the Likert scale mapping dictionary
likert_mapping = {
    'Strongly Disagree': 1,
    'Disagree': 2,
    'Neutral': 3,
    'Agree': 4,
    'Strongly Agree': 5,
    'No Comments': 3 
}

# 3. Apply the mapping to all survey columns
# The first column is 'id. Response ID', so we isolate the question columns
survey_cols = df.columns[1:]
df_numeric = df.copy()

for col in survey_cols:
    df_numeric[col] = df_numeric[col].map(likert_mapping)

# Check for any remaining NaNs that might have been caused by unexpected text
if df_numeric[survey_cols].isna().sum().sum() > 0:
    print("Warning: Missing values detected. Filling with neutral score (3).")
    df_numeric[survey_cols] = df_numeric[survey_cols].fillna(3)

# 4. Extract the final feature matrix for network construction
features = df_numeric[survey_cols].values
respondent_ids = df_numeric['id. Response ID'].values

# Sanity check the resulting matrix
print(f"Successfully vectorized data for {features.shape[0]} respondents across {features.shape[1]} questions.")
display(df_numeric.head())